# Text

> Measuring labels and breaking them over lines, shared by every ItemiSet method.

In [ ]:
#| default_exp core.text

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Names and headers are drawn as live text, so their size comes from the font itself. `text_width` measures a string from the font's outlines, which is what decides how wide a cell or a header must be.

Header labels sometimes need more than one line. `wrap_label` breaks a label only at places a reader would accept: between words first, then at a **soft hyphen** (U+00AD) the label's author put in, then after a dash that is already there. Writing `"Argo\u00adnautes"` in a label says *break here if you must*; the soft hyphen is invisible otherwise.

Dictionary hyphenation is optional (`hyphenate=True`, which needs the `pyphen` package: `pip install itemiset[hyphen]`). It is off by default so that the same labels break the same way on every computer, whether or not `pyphen` is installed.

In [ ]:
#| export
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath

## Measuring

In [ ]:
#| export
def text_width(s: str, size: float, font: str = "DejaVu Sans", weight=400, style="normal") -> float:
    "Advance width of `s` in points, measured from the outlines of `font` at `size` points."
    if not s:
        return 0.0
    tp = TextPath((0, 0), s, size=size, prop=FontProperties(family=font, weight=weight, style=style))
    return tp.get_extents().x1

Widths grow with the text and with the size, and an empty string takes no room:

In [ ]:
test_eq(text_width("", 7), 0.0)
w = text_width("CNOT10", 7)
assert 0 < w < text_width("CNOT10 CNOT10", 7)
test_close(text_width("CNOT10", 14), 2 * w, eps=0.05 * w)
assert text_width("CNOT10", 7, weight=700) > w   # bold is wider

## Breaking labels over lines

In [ ]:
#| export
SHY = "\u00ad"                       # soft hyphen: a break point the label's author chose
DASHES = "-\u2010\u2013\u2014/"     # a line may end after one of these without adding a hyphen

In [ ]:
#| export
def _break_points(marked: str, dictionary) -> list:
    "Positions (in the visible word) where `marked` may be broken."
    clean = marked.replace(SHY, "")
    dash = [i + 1 for i, ch in enumerate(clean[:-1]) if ch in DASHES]
    if SHY in marked:              # the author chose the break points: use only those
        pts, n = [], 0
        for ch in marked:
            if ch == SHY:
                pts.append(n)
            else:
                n += 1
    else:
        pts = dictionary(clean) or list(range(3, len(clean) - 2))   # else: 3+ letters each side
    return sorted({p for p in pts + dash if 0 < p < len(clean)})

def _after(marked: str, p: int) -> str:
    "The part of soft-hyphen-marked `marked` after visible position `p`."
    n = 0
    for i, ch in enumerate(marked):
        if n == p and ch != SHY:
            return marked[i:]
        if ch != SHY:
            n += 1
    return ""

In [ ]:
#| export
def wrap_label(label: str, fits, hyphenate: bool = False) -> list | None:
    """Break `label` into lines that each satisfy `fits(line)`, or return None if a piece cannot fit.

    Whole words move to the next line before any word is broken. A word too long for a line is
    broken at its soft hyphens if it has any; otherwise, with `hyphenate=True`, at dictionary
    points (en_US, via pyphen); otherwise anywhere leaving three or more letters on each side.
    A hyphen is added at the break unless the line already ends with a dash."""
    if hyphenate:
        try:
            import pyphen
        except ImportError as e:
            raise ImportError("hyphenate=True needs pyphen: pip install itemiset[hyphen]") from e
        hy = pyphen.Pyphen(lang="en_US")
        dictionary = lambda w: list(hy.positions(w))
    else:
        dictionary = lambda w: []
    lines, cur = [], ""
    for word in label.split():
        while word:
            clean = word.replace(SHY, "")
            cand = f"{cur} {clean}".strip()
            if fits(cand):
                cur, word = cand, ""
                break
            if cur and fits(clean):      # a whole word on the next line beats a hyphen
                lines.append(cur)
                cur = ""
                continue
            for p in reversed(_break_points(word, dictionary)):
                frag = clean[:p] + ("" if clean[p - 1] in DASHES else "-")
                cand = f"{cur} {frag}".strip()
                if fits(cand):
                    lines.append(cand)
                    cur, word = "", _after(word, p)
                    break
            else:
                if not cur:
                    return None
                lines.append(cur)
                cur = ""
    if cur:
        lines.append(cur)
    return lines

The examples measure lines in characters, so the rules are easy to see: `upto(n)` accepts lines of at most `n` characters.

In [ ]:
upto = lambda n: (lambda t: len(t) <= n)
test_eq(wrap_label("Central metabolism", upto(20)), ["Central metabolism"])
test_eq(wrap_label("Central metabolism", upto(10)), ["Central", "metabolism"])   # whole words first
test_eq(wrap_label("Argo\u00adnautes", upto(6)), ["Argo-", "nautes"])         # the author's break point
test_eq(wrap_label("CCR4–NOT", upto(6)), ["CCR4–", "NOT"])                    # no hyphen after a dash
test_eq(wrap_label("Photosynthesis", upto(9)), ["Photosyn-", "thesis"])        # 3+ letters each side
test_eq(wrap_label("Photosynthesis", upto(3)), None)                           # cannot fit

The soft hyphen never shows unless a line breaks there:

In [ ]:
test_eq(wrap_label("De\u00adcapping complex", upto(20)), ["Decapping complex"])
test_eq(wrap_label("De\u00adcapping complex", upto(8)), ["De-", "capping", "complex"])

In [ ]:
#| hide
try:
    import pyphen
    test_eq(wrap_label("Photosynthesis", lambda t: len(t) <= 9, hyphenate=True), ["Photosyn-", "thesis"])
except ImportError:
    test_fail(lambda: wrap_label("Photosynthesis", lambda t: len(t) <= 9, hyphenate=True), contains="pyphen")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()